# When the clock is wrong: event–tracking timing errors and event-level load

Reproduction notebook for: **Ajjilapuram, Y. (2026). *When the clock is wrong: event–tracking timing errors and their effect on event-level external load monitoring in elite football.* Preprint.**

Runs in Google Colab (free tier). Run all cells **in order**. First run ≈ 45–75 min (download, tracking parsing, ETSY/ELASTIC sync); later runs are faster because data are cached in Google Drive.

**Data:** Sportec Open DFL Dataset (Bassek et al., 2025; CC-BY 4.0) and the ELASTIC annotated benchmark (Kim et al., 2026; data CC-BY 4.0, code MPL-2.0). Data are downloaded by the notebook, not stored in this repository.

## 1. Setup – mount Drive, get ELASTIC code, install packages
*Run every session.*

In [ ]:
from google.colab import drive
import os, sys, subprocess

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/paper1'
os.makedirs(BASE, exist_ok=True)
os.chdir(BASE)
if not os.path.exists(f'{BASE}/elastic/tools/sportec_data.py'):
    subprocess.run(['git', 'clone', 'https://github.com/hyunsungkim-ds/elastic.git'], check=True)

ROOT = f'{BASE}/elastic'
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
subprocess.run('pip install -q kloppy pandera py7zr fastparquet pyarrow scipy tqdm', shell=True)

MATCHES = ['J03WMX', 'J03WN1', 'J03WPY']
print('Python', sys.version.split()[0])
print('Now in:', os.getcwd())
print('tools found:', os.path.exists('tools/sportec_data.py'))
print('benchmark:', sorted(os.listdir('benchmark/gt')))


## 2. Download the three benchmark matches from figshare (once)

In [ ]:
import glob, shutil, zipfile, requests
RAW = 'data/raw'; os.makedirs(RAW, exist_ok=True)

files = requests.get('https://api.figshare.com/v2/articles/28196177/files?page_size=100', timeout=60).json()
print(len(files), 'files on figshare. First few:')
for f in files[:12]:
    print('  ', f['name'], round(f['size']/1e6, 1), 'MB')

named = [f for f in files if any(m in f['name'] for m in MATCHES)]
todo = named if named else files
print('\nWill download', len(todo), 'file(s)')

for f in todo:
    path = os.path.join(RAW, f['name'])
    if os.path.exists(path) and os.path.getsize(path) == f['size']:
        print('have', f['name']); continue
    print('downloading', f['name'], '...')
    with requests.get(f['download_url'], stream=True, timeout=900) as r:
        r.raise_for_status()
        with open(path, 'wb') as out:
            for chunk in r.iter_content(1 << 20):
                out.write(chunk)
print('download done')


## 3. Extract and sort XML files into ELASTIC's folder layout (once)

In [ ]:
for a in glob.glob(f'{RAW}/*'):
    if a.endswith('.zip'):
        zipfile.ZipFile(a).extractall(f'{RAW}/extracted')
    elif a.endswith('.7z'):
        import py7zr
        with py7zr.SevenZipFile(a) as z:
            z.extractall(f'{RAW}/extracted')

DEST = {'matchinformation': 'data/sportec/metadata',
        'events': 'data/sportec/event',
        'positions': 'data/sportec/tracking'}
for d in DEST.values():
    os.makedirs(d, exist_ok=True)
for x in glob.glob(f'{RAW}/**/*.xml', recursive=True):
    n = os.path.basename(x)
    for key, d in DEST.items():
        if key in n.lower() and not os.path.exists(os.path.join(d, n)):
            shutil.copy(x, os.path.join(d, n))

ok = True
for key, d in DEST.items():
    got = [m for m in MATCHES if any(m in f for f in os.listdir(d))]
    print(f'{d:28s} matches found: {got}')
    ok &= len(got) == 3
print('\nALL GOOD' if ok else '\nSOMETHING MISSING – send Claude the output of Cells 2 and 3')


## 4. Parse tracking once and cache as parquet (once)

In [ ]:
import importlib
import tools.sportec_data as sd
importlib.reload(sd)
os.makedirs('data/sportec/tracking_parquet', exist_ok=True)

sd.TRACKING_DIR = 'data/sportec/tracking'            # read XML this time
for m in MATCHES:
    out = f'data/sportec/tracking_parquet/{m}.parquet'
    if os.path.exists(out):
        print('cached', m); continue
    print('reading', m, '...')
    md_ = sd.SportecData(m)
    md_.tracking.to_parquet(out)
    print('saved', m, md_.tracking.shape)
    del md_
sd.TRACKING_DIR = 'data/sportec/tracking_parquet'    # fast mode
print('done')


## 5. Inspect the benchmark (raw vs annotated event frames)

In [ ]:
import pandas as pd
raw0 = pd.read_parquet('benchmark/unsynced/J03WMX.parquet')
gt0 = pd.read_parquet('benchmark/gt/J03WMX.parquet')
print('raw', raw0.shape, '| gt', gt0.shape)
print('raw columns:', raw0.columns.tolist())
print('gt columns:', gt0.columns.tolist())
display(raw0.head(3)); display(gt0.head(3))


## 6. Core functions – timing error and event-level load
*Run every session.* Note: `flags()` here contains a pilot instantaneous peak-acceleration flag that was **not used** in the paper (it fired on 82% of events); the final flags are defined in Section 11.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tools.sportec_data as sd
sd.TRACKING_DIR = 'data/sportec/tracking_parquet'
from tools.sportec_data import SportecData

FPS = 25
WINDOW_S = 2.0        # seconds before and after each event
HSR_SPEED = 5.5       # m/s, illustrative high-speed threshold
SPRINT_SPEED = 7.0    # m/s, illustrative sprint threshold
HIGH_ACCEL = 3.0      # m/s^2, illustrative high-acceleration threshold
OUT_DIR = 'paper1_results'
os.makedirs(OUT_DIR, exist_ok=True)
LOAD_KEYS = ['dist_m', 'mean_speed', 'peak_speed', 'peak_accel', 'hsr_m']


def to_utc(s):
    s = pd.to_datetime(s)
    try:
        return s.dt.tz_convert('UTC').dt.tz_localize(None) if s.dt.tz is not None else s
    except AttributeError:
        return s


def nearest_frames(frames, periods, stamps):
    """Fast nearest-frame lookup for many events (same period only)."""
    out = np.full(len(stamps), np.nan)
    for p, f in frames.groupby('period_id'):
        f = f.sort_values('utc_timestamp')
        t = f['utc_timestamp'].values.astype('datetime64[ns]').astype('int64')
        ids = f['frame_id'].values
        mask = (periods == p) & ~pd.isna(stamps)
        if not mask.any():
            continue
        q = stamps[mask].values.astype('datetime64[ns]').astype('int64')
        j = np.clip(np.searchsorted(t, q), 1, len(t) - 1)
        left_closer = (q - t[j - 1]) <= (t[j] - q)
        out[mask] = np.where(left_closer, ids[j - 1], ids[j])
    return out


def window_load(track, centre):
    if pd.isna(centre):
        return {k: np.nan for k in LOAD_KEYS}
    w = int(WINDOW_S * FPS)
    seg = track[(track['frame_id'] >= centre - w) & (track['frame_id'] <= centre + w)]
    if seg.empty:
        return {k: np.nan for k in LOAD_KEYS}
    step = seg['speed'] / FPS
    return {'dist_m': step.sum(), 'mean_speed': seg['speed'].mean(),
            'peak_speed': seg['speed'].max(), 'peak_accel': seg['accel_s'].abs().max(),
            'hsr_m': step[seg['speed'] >= HSR_SPEED].sum()}


def flags(load):
    f = {}
    for name, key, thr in [('flag_sprint', 'peak_speed', SPRINT_SPEED), ('flag_accel', 'peak_accel', HIGH_ACCEL)]:
        f[name] = np.nan if pd.isna(load[key]) else bool(load[key] >= thr)
    return f


def run_match(mid):
    print(f'\n=== {mid} ===')
    raw = pd.read_parquet(f'benchmark/unsynced/{mid}.parquet')
    gt = pd.read_parquet(f'benchmark/gt/{mid}.parquet')
    data = SportecData(mid)
    tracking = data.format_tracking_for_syncer()
    tracking['utc_timestamp'] = to_utc(tracking['utc_timestamp'])
    raw['utc_timestamp'] = to_utc(raw['utc_timestamp'])

    players = tracking[~tracking['ball'].astype(bool)]
    frames = tracking[['frame_id', 'period_id', 'utc_timestamp']].drop_duplicates('frame_id')
    id_map = data.lineup.set_index('player_id')['object_id'].to_dict()
    tracks = {pid: g.sort_values('frame_id') for pid, g in players.groupby('player_id')}

    common = raw.index.intersection(gt.index)
    raw, gt = raw.loc[common], gt.loc[common]
    raw_frames = nearest_frames(frames, raw['period_id'].values, raw['utc_timestamp'])

    rows = []
    for k, i in enumerate(common):
        pid = raw.at[i, 'player_id']
        obj = pid if pid in tracks else id_map.get(pid)
        rf, tf = raw_frames[k], gt.at[i, 'frame_id']
        if obj not in tracks or pd.isna(rf) or pd.isna(tf):
            continue
        lr, lt = window_load(tracks[obj], rf), window_load(tracks[obj], tf)
        fr, ft = flags(lr), flags(lt)
        row = {'match_id': mid, 'event_idx': i, 'player': obj,
               'spadl_type': raw.at[i, 'spadl_type'] if 'spadl_type' in raw.columns else None,
               'raw_frame': rf, 'true_frame': tf, 'timing_error_s': (rf - tf) / FPS}
        for key in LOAD_KEYS:
            row[f'{key}_raw'], row[f'{key}_true'] = lr[key], lt[key]
            row[f'{key}_diff'] = lr[key] - lt[key]
        for key in fr:
            row[f'{key}_raw'], row[f'{key}_true'] = fr[key], ft[key]
            row[f'{key}_flip'] = fr[key] != ft[key]
        rows.append(row)
    df = pd.DataFrame(rows)
    print(f'events in benchmark: {len(common)} | analysed: {len(df)}')
    return df, tracks


def summarise(df):
    err = df['timing_error_s']
    print('\n--- Timing error (raw - true), seconds ---')
    print(f'mean {err.mean():.2f} | SD {err.std():.2f} | median |err| {err.abs().median():.2f} | max |err| {err.abs().max():.2f}')
    for t in [0.2, 0.5, 1.0, 2.0]:
        print(f'  |error| > {t}s: {(err.abs() > t).mean()*100:.1f}% of events')
    print('\n--- Change in event-level load (raw vs true) ---')
    for key in ['dist_m', 'peak_speed', 'peak_accel', 'hsr_m']:
        d = df[f'{key}_diff'].abs()
        print(f'  {key:10s}: median |diff| {d.median():.2f} | 90th pct {d.quantile(0.9):.2f}')
    print('\n--- Decision flips (illustrative flags) ---')
    for key in ['flag_sprint', 'flag_accel']:
        print(f'  {key}: {df[key + "_flip"].mean()*100:.1f}% of events flip')
    if df['spadl_type'].notna().any():
        print('\n--- Median |timing error| by event type (top 10 by count) ---')
        g = df.groupby('spadl_type')['timing_error_s'].agg(n='size', med_abs=lambda s: s.abs().median())
        print(g.sort_values('n', ascending=False).head(10).round(2))


def figures(df, tracks, tag):
    plt.figure(figsize=(6, 4))
    plt.hist(df['timing_error_s'].clip(-5, 5), bins=80)
    plt.xlabel('Timing error: raw minus true (s), clipped to ±5 s')
    plt.ylabel('Number of events'); plt.title(f'Event timing error ({tag})')
    plt.tight_layout(); plt.savefig(f'{OUT_DIR}/fig1_timing_error_{tag}.png', dpi=200); plt.show()

    sub = df[df['player'].isin(tracks.keys())]
    if len(sub):
        ex = sub.loc[sub['timing_error_s'].abs().idxmax()]
        tr = tracks[ex['player']]
        lo = min(ex['raw_frame'], ex['true_frame']) - 100
        hi = max(ex['raw_frame'], ex['true_frame']) + 100
        seg = tr[(tr['frame_id'] >= lo) & (tr['frame_id'] <= hi)]
        plt.figure(figsize=(7, 3.5))
        plt.plot(seg['frame_id'] / FPS, seg['speed'], lw=1.2)
        plt.axvline(ex['true_frame'] / FPS, ls='-', label='true event time')
        plt.axvline(ex['raw_frame'] / FPS, ls='--', label='raw event time')
        plt.xlabel('Match time (s)'); plt.ylabel('Player speed (m/s)')
        plt.title(f'{ex["match_id"]}: speed around a large-error event'); plt.legend()
        plt.tight_layout(); plt.savefig(f'{OUT_DIR}/fig2_example_event_{tag}.png', dpi=200); plt.show()

print('functions ready')


## 7. Raw-feed analysis for all three matches

In [ ]:
allres = []
for m in MATCHES:
    r, t = run_match(m)
    allres.append(r)
    del t
res_all = pd.concat(allres, ignore_index=True)
res_all.to_csv(f'{OUT_DIR}/event_load_all.csv', index=False)
summarise(res_all)


## 8. Run the synchronisation methods (ETSY, ELASTIC) – once, ~10–40 min
Uses the ELASTIC repository's benchmark script; outputs are cached in `benchmark/synced/`.

In [ ]:
import os
for meth in ['etsy', 'elastic_nw']:
    if all(os.path.exists(f'benchmark/synced/{meth}/{m}.parquet') for m in MATCHES):
        print('cached', meth); continue
    !python experiments/evaluate.py --method {meth} --save


## 9. Timestamp conditions: raw, best constant shift, ETSY, ELASTIC

In [ ]:
# Methods to compare: raw, best constant shift, ELASTIC (if available)
per = []
for mid in MATCHES:
    g = pd.read_parquet(f'benchmark/gt/{mid}.parquet')[['period_id']]
    g['match_id'] = mid; g['event_idx'] = g.index
    per.append(g)
R = res_all.drop(columns=[c for c in ['period_id'] if c in res_all]).merge(
    pd.concat(per), on=['match_id', 'event_idx'], how='left')
R['err_frames'] = R['raw_frame'] - R['true_frame']
# best possible constant clock shift per match-half (upper bound for simple fixes)
R['shift_frame'] = R['raw_frame'] - R.groupby(['match_id', 'period_id'])['err_frames'].transform('median').round()

END = ['control', 'out', 'goal']
for meth, colname in [('etsy', 'etsy_frame'), ('elastic_nw', 'elastic_frame')]:
    R[colname] = np.nan
    for mid in MATCHES:
        p = f'benchmark/synced/{meth}/{mid}.parquet'
        if not os.path.exists(p):
            continue
        s = pd.read_parquet(p)
        s = s[~s['spadl_type'].isin(END)].reset_index(drop=True)
        g = pd.read_parquet(f'benchmark/gt/{mid}.parquet').reset_index(drop=True)
        if len(s) != len(g) or (s['spadl_type'].values != g['spadl_type'].values).mean() > 0.01:
            print(f'WARNING {meth} {mid}: events do not line up ({len(s)} vs {len(g)}) - skipped'); continue
        m = R['match_id'] == mid
        R.loc[m, colname] = R.loc[m, 'event_idx'].map(s['frame_id']).values

METHODS = {'Raw feed': 'raw_frame', 'Constant shift': 'shift_frame', 'ETSY': 'etsy_frame', 'ELASTIC': 'elastic_frame'}
METHODS = {k: v for k, v in METHODS.items() if R[v].notna().any()}
print('methods available:', list(METHODS))
for name, col in METHODS.items():
    e = ((R[col] - R['true_frame']) / FPS).abs()
    print(f'{name:15s} median |err| {e.median():.2f}s | >1s {(e > 1).mean()*100:.1f}% | >2s {(e > 2).mean()*100:.1f}%')


## 10. Peak speed / acceleration in ±1, ±2, ±3 s windows for every condition

In [ ]:
# Fast window metrics for every method, window and threshold
def np_tracks(mid):
    data = SportecData(mid)
    tr = data.format_tracking_for_syncer()
    tr = tr[~tr['ball'].astype(bool)]
    out = {}
    for pid, g in tr.groupby('player_id'):
        g = g.sort_values('frame_id')
        out[pid] = (g['frame_id'].values, g['speed'].values, np.abs(g['accel_s'].values))
    return out

def peaks(trk, c, win):
    f, sp, ac = trk
    w = int(win * FPS)
    a, b = np.searchsorted(f, c - w), np.searchsorted(f, c + w, side='right')
    if b <= a:
        return np.nan, np.nan
    return np.nanmax(sp[a:b]), np.nanmax(ac[a:b])

WINDOWS = [1.0, 2.0, 3.0]
cols = ['true_frame'] + list(METHODS.values())
recs = []
for mid in MATCHES:
    T = np_tracks(mid)
    for i, r in R[R['match_id'] == mid].iterrows():
        trk = T.get(r['player'])
        if trk is None:
            continue
        rec = {'i': i}
        for win in WINDOWS:
            for col in cols:
                if pd.notna(r[col]):
                    rec[f'ps|{col}|{win}'], rec[f'pa|{col}|{win}'] = peaks(trk, r[col], win)
        recs.append(rec)
    del T
S = pd.DataFrame(recs).set_index('i')
R = R.join(S)

FLAGS = {'High acceleration': ('pa', [2.0, 2.5, 3.0, 3.5, 4.0]),
         'Sprint': ('ps', [6.0, 6.5, 7.0, 7.5, 8.0])}
rows = []
for win in WINDOWS:
    for fname, (pre, thrs) in FLAGS.items():
        for thr in thrs:
            truth_v = R[f'{pre}|true_frame|{win}']
            for mname, col in METHODS.items():
                pred_v = R[f'{pre}|{col}|{win}']
                ok = truth_v.notna() & pred_v.notna()
                t, p = truth_v[ok] >= thr, pred_v[ok] >= thr
                rows.append(dict(window_s=win, flag=fname, threshold=thr, method=mname, n=int(ok.sum()),
                                 base_rate=t.mean()*100, flip=(t != p).mean()*100,
                                 false_alert=(p & ~t).mean()*100, missed=(t & ~p).mean()*100,
                                 missed_of_true=(t & ~p).sum() / max(t.sum(), 1) * 100))
TAB = pd.DataFrame(rows).round(1)
TAB.to_csv(f'{OUT_DIR}/table_flags_all_settings.csv', index=False)
print('Saved all-settings table:', f'{OUT_DIR}/table_flags_all_settings.csv')


## 11. Figure style

In [ ]:
# Publication figures (PDF + PNG, colour-blind-safe, consistent style)
import matplotlib as mpl
mpl.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.color': '#e6e6e6',
                     'grid.linewidth': 0.6, 'axes.edgecolor': '#555555', 'legend.frameon': False,
                     'savefig.bbox': 'tight'})
COL = {'Raw feed': '#0072B2', 'Constant shift': '#E69F00', 'ETSY': '#CC79A7', 'ELASTIC': '#009E73'}
LS = {'Raw feed': '-', 'Constant shift': '--', 'ETSY': '-.', 'ELASTIC': ':'}
def save(fig, name):
    fig.savefig(f'{OUT_DIR}/{name}.pdf'); fig.savefig(f'{OUT_DIR}/{name}.png', dpi=300); plt.show()


## 12. Main analysis (paper Table 2, event-type table, sprint figure)
Same events for all conditions; sustained-acceleration flags; bootstrap CIs; exact McNemar tests vs raw feed.

In [ ]:
from scipy.stats import binomtest

# 1) Same events for every method (ETSY left some events unsynced)
common = R[[c for c in METHODS.values()] + ['true_frame']].notna().all(axis=1)
RC = R[common].copy()
print(f'Common events for all methods: {len(RC)} of {len(R)}')

# 2) Sustained acceleration effort: accel >= thr for >= 0.5 s inside the window
MIN_RUN = int(0.5 * FPS)
def sustained(trk, c, win, thr):
    f, sp, ac_abs = trk
    w = int(win * FPS)
    a, b = np.searchsorted(f, c - w), np.searchsorted(f, c + w, side='right')
    if b <= a:
        return np.nan
    x = ACC_SIGNED[id(trk)][a:b] >= thr
    best = run = 0
    for v in x:
        run = run + 1 if v else 0
        best = max(best, run)
    return float(best >= MIN_RUN)

def np_tracks_signed(mid):
    data = SportecData(mid)
    tr = data.format_tracking_for_syncer()
    tr = tr[~tr['ball'].astype(bool)]
    out = {}
    for pid, g in tr.groupby('player_id'):
        g = g.sort_values('frame_id')
        trk = (g['frame_id'].values, g['speed'].values, np.abs(g['accel_s'].values))
        ACC_SIGNED[id(trk)] = g['accel_s'].values
        out[pid] = trk
    return out

ACC_THRS = [2.0, 2.5, 3.0]
cols = ['true_frame'] + list(METHODS.values())
recs = []
for mid in MATCHES:
    ACC_SIGNED = {}
    T = np_tracks_signed(mid)
    for i, r in RC[RC['match_id'] == mid].iterrows():
        trk = T.get(r['player'])
        if trk is None:
            continue
        rec = {'i': i}
        for thr in ACC_THRS:
            for col in cols:
                rec[f'sa|{col}|{thr}'] = sustained(trk, r[col], 2.0, thr)
        recs.append(rec)
    del T
RC = RC.join(pd.DataFrame(recs).set_index('i'))

# 3) Table with 95% CIs (match-stratified bootstrap) and McNemar-type test vs raw
rng = np.random.default_rng(7)
def flag_pair(df, kind, col, thr, win=2.0):
    if kind == 'sa':
        a, b = df[f'sa|true_frame|{thr}'], df[f'sa|{col}|{thr}']
        ok = a.notna() & b.notna(); return a[ok] >= 1, b[ok] >= 1
    a, b = df[f'{kind}|true_frame|{win}'], df[f'{kind}|{col}|{win}']
    ok = a.notna() & b.notna(); return a[ok] >= thr, b[ok] >= thr

def metrics(df, kind, col, thr):
    t, p = flag_pair(df, kind, col, thr)
    return (t != p).mean()*100, (t & ~p).sum()/max(t.sum(), 1)*100, (p & ~t).sum()/max(p.sum(), 1)*100, t.mean()*100

SPECS = [('Sprint ≥7 m/s', 'ps', 7.0), ('High-speed ≥5.5 m/s', 'ps', 5.5),
         ('Sustained accel ≥2.5 m/s² for ≥0.5 s', 'sa', 2.5), ('Sustained accel ≥3 m/s² for ≥0.5 s', 'sa', 3.0)]
out = []
for fname, kind, thr in SPECS:
    t_raw, p_raw = flag_pair(RC, kind, 'raw_frame', thr)
    for mname, col in METHODS.items():
        flip, missed, false_alert, base = metrics(RC, kind, col, thr)
        bs = []
        for _ in range(1000):
            parts = [RC[RC.match_id == m].sample(frac=1, replace=True, random_state=int(rng.integers(1e9))) for m in MATCHES]
            bs.append(metrics(pd.concat(parts), kind, col, thr)[1])
        row = dict(flag=fname, method=mname, base_rate=base, flip=flip, missed_true_flags=missed,
                   missed_CI=f'{np.percentile(bs, 2.5):.1f}–{np.percentile(bs, 97.5):.1f}',
                   false_alerts_of_raised=false_alert)
        if mname != 'Raw feed':
            t, p = flag_pair(RC, kind, col, thr)
            wrong_raw, wrong_m = (t_raw != p_raw), (t != p)
            b_, c_ = int((wrong_raw & ~wrong_m).sum()), int((~wrong_raw & wrong_m).sum())
            row['p_vs_raw'] = binomtest(b_, b_ + c_, 0.5).pvalue if b_ + c_ else np.nan
        out.append(row)
FINAL = pd.DataFrame(out)
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 20)
display(FINAL.round(3))
FINAL.to_csv(f'{OUT_DIR}/Table2_flags_fair_comparison.csv', index=False)

# 4) Timing error by event type and method (common events)
ET = RC.assign(**{n: ((RC[c] - RC['true_frame']) / FPS).abs() for n, c in METHODS.items()})
top = ET['spadl_type'].value_counts().head(8).index
T3 = ET[ET.spadl_type.isin(top)].groupby('spadl_type')[list(METHODS)].median().round(2)
T3.insert(0, 'n', ET[ET.spadl_type.isin(top)].spadl_type.value_counts())
display(T3.sort_values('n', ascending=False))
T3.to_csv(f'{OUT_DIR}/Table3_error_by_event_type.csv')

# 5) Fig 2 v2 – sprint missed rate by method (with 95% CI)
fig, ax = plt.subplots(figsize=(4.4, 2.8))
sub = FINAL[FINAL.flag == 'Sprint ≥7 m/s'].reset_index(drop=True)
lo_hi = sub['missed_CI'].str.split('–', expand=True).astype(float)
y = sub['missed_true_flags']
ax.barh(sub['method'], y, color=[COL[m] for m in sub['method']], height=0.6)
ax.errorbar(y, range(len(sub)), xerr=[y - lo_hi[0], lo_hi[1] - y], fmt='none', ecolor='#333333', capsize=3, lw=1)
for k, v in enumerate(y):
    ax.text(lo_hi[1][k] + 0.8, k, f'{v:.1f}%', va='center', fontsize=8)
ax.invert_yaxis(); ax.set_xlabel('True sprint-linked events missed (%)')
ax.set_title('Sprint flags missed at each sync level', loc='left', fontsize=9)
ax.set_xlim(0, max(lo_hi[1]) * 1.25); ax.grid(axis='y', visible=False)
save(fig, 'Fig5_sprint_missed_by_method')

print('Done: Table2, Table3, Fig5 saved in', OUT_DIR)


## 13. Dose–response, GEE, per-match and sensitivity analyses (paper Figure 5, Tables S1–S3, Figure S1)

In [ ]:
import subprocess; subprocess.run('pip install -q statsmodels', shell=True)
import statsmodels.api as sm, statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportion_confint
import matplotlib as mpl
mpl.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 7.5, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#888888', 'axes.linewidth': 0.6, 'xtick.color': '#666666', 'ytick.color': '#666666',
    'axes.grid': True, 'grid.color': '#E9E9E9', 'grid.linewidth': 0.5, 'legend.frameon': False,
    'savefig.bbox': 'tight', 'pdf.fonttype': 42})

# A) Per-match consistency (missed % of true flags)
SP = [('Sprint', 'ps', 7.0), ('Sustained accel 3.0', 'sa', 3.0)]
pm = []
for mid in MATCHES:
    d = RC[RC.match_id == mid]
    for fname, kind, thr in SP:
        for mname, col in METHODS.items():
            t, p = flag_pair(d, kind, col, thr)
            pm.append(dict(match=mid, flag=fname, method=mname, n_true=int(t.sum()),
                           missed=round((t & ~p).sum() / max(t.sum(), 1) * 100, 1)))
PM = pd.DataFrame(pm).pivot_table(index=['flag', 'method'], columns='match', values='missed')
print('A) Missed % by match'); display(PM)
PM.to_csv(f'{OUT_DIR}/TableS1_per_match.csv')

# B) Dose-response: missed % of true flags by size of raw timing error (Wilson 95% CI)
RC['abs_err_s'] = ((RC['raw_frame'] - RC['true_frame']) / FPS).abs()
bins = [0, 0.2, 0.5, 1, 2, 4, np.inf]; labels = ['<0.2', '0.2–0.5', '0.5–1', '1–2', '2–4', '≥4']
RC['err_bin'] = pd.cut(RC['abs_err_s'], bins, labels=labels, right=False)
dr = []
for fname, kind, thr in SP:
    for b in labels:
        d = RC[RC.err_bin == b]
        t, p = flag_pair(d, kind, 'raw_frame', thr)
        k, n = int((t & ~p).sum()), int(t.sum())
        lo, hi = proportion_confint(k, n, method='wilson') if n else (np.nan, np.nan)
        dr.append(dict(flag=fname, bin=b, n_events=len(d), n_true=n, missed=k / n * 100 if n else np.nan, lo=lo * 100, hi=hi * 100))
DR = pd.DataFrame(dr).round(1); print('\nB) Dose-response'); display(DR)
DR.to_csv(f'{OUT_DIR}/TableS2_dose_response.csv', index=False)

fig, ax = plt.subplots(figsize=(3.35, 2.5))
x = np.arange(len(labels))
for k, (fname, col, mk) in enumerate([('Sprint', '#0072B2', 'o'), ('Sustained accel 3.0', '#E69F00', 's')]):
    d = DR[DR.flag == fname]
    xx = x + (k - 0.5) * 0.18
    ax.plot(xx, d.missed, marker=mk, color=col, lw=1.4, ms=4.5, mec='white', mew=0.6,
            label='Sprint (≥7.0 m·s⁻¹)' if k == 0 else 'Sustained acceleration (≥3.0 m·s⁻², ≥0.5 s)')
    ax.vlines(xx, d.lo, d.hi, color=col, lw=1)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_xlabel('Absolute timing error of raw timestamp (s)'); ax.set_ylabel('True flags missed (%, 95% CI)')
ax.set_ylim(0, 100); ax.legend(loc='upper left', fontsize=6.4)
for i, n in enumerate(DR[DR.flag == 'Sprint'].n_events):
    ax.text(i, -14, f'n={n}', ha='center', fontsize=5.8, color='#666666', clip_on=False)
fig.savefig(f'{OUT_DIR}/Fig4_dose_response.pdf'); fig.savefig(f'{OUT_DIR}/Fig4_dose_response.png', dpi=400); plt.show()

# C) GEE logistic regression: odds of a miss per 1 s of error, clustered by player (true-flag events only)
gee_rows = []
for fname, kind, thr in SP:
    t, p = flag_pair(RC, kind, 'raw_frame', thr)
    d = RC.loc[t.index[t]].copy(); d['miss'] = (~p[t]).astype(int).values
    d['cluster'] = d['match_id'] + '_' + d['player'].astype(str)
    mod = smf.gee('miss ~ abs_err_s', groups='cluster', data=d, family=sm.families.Binomial(),
                  cov_struct=sm.cov_struct.Exchangeable()).fit()
    orr = np.exp(mod.params['abs_err_s']); ci = np.exp(mod.conf_int().loc['abs_err_s'])
    gee_rows.append(dict(flag=fname, n=len(d), clusters=d.cluster.nunique(), OR_per_s=orr, lo=ci[0], hi=ci[1], p=mod.pvalues['abs_err_s']))
GEE = pd.DataFrame(gee_rows); print('\nC) GEE'); display(GEE.round(4))
GEE.to_csv(f'{OUT_DIR}/TableS3_gee.csv', index=False)

# D) Sensitivity: sprint misses across windows and thresholds (raw vs ELASTIC)
sens = []
for win in [1.0, 2.0, 3.0]:
    for thr in [6.0, 6.5, 7.0, 7.5, 8.0]:
        for mname in [m for m in ['Raw feed', 'ELASTIC'] if m in METHODS]:
            col = METHODS[mname]
            a, b = RC[f'ps|true_frame|{win}'], RC[f'ps|{col}|{win}']
            ok = a.notna() & b.notna(); t, p = a[ok] >= thr, b[ok] >= thr
            sens.append(dict(window=win, thr=thr, method=mname, missed=(t & ~p).sum() / max(t.sum(), 1) * 100))
SE = pd.DataFrame(sens)
SM = [m for m in ['Raw feed', 'ELASTIC'] if m in METHODS]
fig, axes = plt.subplots(1, len(SM), figsize=(3.4 * len(SM), 2.0), gridspec_kw=dict(wspace=0.25), squeeze=False)
axes = axes[0]
for ax, mname in zip(axes, SM):
    H = SE[SE.method == mname].pivot(index='window', columns='thr', values='missed')
    ax.imshow(H.values, cmap='Blues', vmin=0, vmax=SE.missed.max(), aspect='auto')
    for (i, j), v in np.ndenumerate(H.values):
        ax.text(j, i, f'{v:.1f}', ha='center', va='center', fontsize=6.8, color='white' if v > SE.missed.max() * 0.6 else '#222222')
    ax.set_xticks(range(H.shape[1])); ax.set_xticklabels([f'{c:g}' for c in H.columns])
    ax.set_yticks(range(H.shape[0])); ax.set_yticklabels([f'±{w:g} s' for w in H.index])
    ax.set_xlabel('Sprint threshold (m·s⁻¹)'); ax.grid(False); ax.set_title(mname, loc='left', fontsize=8, fontweight='bold')
axes[0].set_ylabel('Window')
fig.savefig(f'{OUT_DIR}/FigS1_sensitivity.pdf'); fig.savefig(f'{OUT_DIR}/FigS1_sensitivity.png', dpi=400); plt.show()
display(SE.pivot_table(index=['method', 'window'], columns='thr', values='missed').round(1))
SE.to_csv(f'{OUT_DIR}/TableS4_sensitivity.csv', index=False)

# E) Descriptives for Methods section
print('\nE) Descriptives')
print('events per match:', R.groupby('match_id').size().to_dict())
print('players with events:', R.player.nunique(), '| ETSY unsynced events:', int(R['etsy_frame'].isna().sum()))
e = (R['raw_frame'] - R['true_frame']) / FPS
print(f'raw signed error: mean {e.mean():.2f} s, SD {e.std():.2f}, median {e.median():.2f}, IQR {e.quantile(.25):.2f} to {e.quantile(.75):.2f}; '
      f'% late (raw after true) {(e > 0).mean()*100:.1f}%')
print('\nAll tables saved in', OUT_DIR)


## Outputs
All tables (`.csv`) and figures (`.pdf`, `.png`) are written to `paper1_results/`. Final publication figures are re-drawn from these numbers with `figures/make_figs.py` and `figures/make_figs2.py` in this repository.